In [ ]:
pip install earthengine-api geemap

In [ ]:
import ee
import datetime

ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

# Brazil bounding box
bbox = ee.Geometry.BBox(-94.1875, -39.0208, 37.0625, 18.2292)

# Load SPEI dataset (12-month scale)
spei = ee.ImageCollection('CSIC/SPEI/2_10').select('SPEI_12_month').filterBounds(bbox)

# Define year range - UPDATE FOR NEW YEARS
start_year = 2025
end_year = 2025

print(f"Exporting SPEI data for Brazil: {start_year}-{end_year}")
print(f"Bounding box: {bbox.getInfo()['coordinates']}")

In [ ]:
# Export monthly SPEI data
for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        # Create date range for this month
        start_date = datetime.date(year, month, 1)
        
        # Handle end of month
        if month == 12:
            end_date = datetime.date(year + 1, 1, 1)
        else:
            end_date = datetime.date(year, month + 1, 1)
        
        # Filter to specific month
        month_collection = spei.filterDate(
            start_date.isoformat(),
            end_date.isoformat()
        )
        
        # Get the image for this month (should be single image)
        month_img = month_collection.first().clip(bbox)
        
        # Create filename
        month_str = f"{month:02d}"
        filename = f"spei_{year}_{month_str}"
        
        # Export to Drive
        task = ee.batch.Export.image.toDrive(
            image=month_img,
            description=filename,
            folder="Brazil_SPEI_Monthly",
            fileNamePrefix=filename,
            region=bbox,
            scale=55000,  # Native resolution ~55km
            maxPixels=1e13,
            crs='EPSG:4326'
        )
        
        task.start()
        print(f"Exporting: {filename}")

print("\nAll SPEI export tasks initiated!")
print("Check Google Earth Engine Tasks tab for progress.")